In [ ]:
# Cell 1: Dependencies
#!pip install google-generativeai datasets pandas pyarrow huggingface_hub scikit-learn

# Cell 2: Imports
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset, DatasetDict
import os
import json
import time
import google.generativeai as genai
from datetime import datetime
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import hashlib
from huggingface_hub import login, upload_file, hf_hub_download

In [ ]:
# Cell 3: Configuration
CONFIG = {
    #'gemini_api_key': '',
    'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    
    #'gemini_api_key': '',
    #'gemini_api_key': '',
    'huggingface_token': '',
    'original_dataset': 'Azzindani/Indonesian_Legal_QA',
    'output_repository': 'Azzindani/ID_Legal_Bench',
    'question_column': 'Question',
    'answer_column': 'Answer',
    'context_column': None,
    'citation_column': None,
    'preserve_columns': [],
    
    # Processing settings
    'chunk_size': 20,
    'num_variants': 5,  # Adjustable number of variants
    'model_name': 'gemini-2.5-flash',
    'temperature': 0.6,
    'input_token_limit': 10000,
    'output_token_limit': lambda: CONFIG['num_variants'] * 4000,  # variants x 4000
    
    # Quality settings
    'min_question_length': 15,
    'min_answer_length': 50,
    'max_question_length': 2000,
    'max_answer_length': 8000,
    'min_overall_score': 0.60,
    
    # File structure
    'data_dir': './data/',
    'progress_file': 'progress.json'  # This will be in root of output repo
}

# Create directories
os.makedirs(CONFIG['data_dir'], exist_ok=True)

In [ ]:
# Cell 4: Authentication
genai.configure(api_key=CONFIG['gemini_api_key'])
login(token=CONFIG['huggingface_token'])

In [ ]:
# Cell 5: Progress Manager
class ProgressManager:
    def __init__(self):
        self.progress_data = {
            'processed_indices': [],
            'current_chunk': 0,
            'total_chunks': 0,
            'total_processed': 0,
            'total_variants_created': 0,
            'request_count': 0,
            'start_time': None,
            'last_update': None,
            'errors': [],
            'statistics': {
                'quality_counts': {'premium': 0, 'high': 0, 'medium': 0, 'low': 0, 'poor': 0},
                'avg_score': 0.0,
                'avg_word_count': 0.0
            }
        }
        self.load_progress()
    
    def load_progress(self):
        """Load progress from output repository"""
        try:
            progress_path = hf_hub_download(
                repo_id=CONFIG['output_repository'],
                filename=CONFIG['progress_file'],
                repo_type="dataset"
            )
            with open(progress_path, 'r') as f:
                saved_progress = json.load(f)
                self.progress_data.update(saved_progress)
            print(f"✅ Progress loaded: {len(self.progress_data['processed_indices'])} processed")
        except Exception as e:
            print(f"⚠️ No existing progress found, starting fresh: {e}")
            self.progress_data['start_time'] = datetime.now().isoformat()
    
    def save_progress(self):
        """Save progress to output repository"""
        try:
            self.progress_data['last_update'] = datetime.now().isoformat()
            
            # Convert any numpy/pandas types to native Python types
            def convert_types(obj):
                if isinstance(obj, dict):
                    return {k: convert_types(v) for k, v in obj.items()}
                elif isinstance(obj, list):
                    return [convert_types(v) for v in obj]
                elif hasattr(obj, 'item'):  # numpy types
                    return obj.item()
                elif hasattr(obj, 'tolist'):  # numpy arrays
                    return obj.tolist()
                else:
                    return obj
            
            # Clean the data before JSON serialization
            clean_data = convert_types(self.progress_data)
            
            # Save locally first
            local_path = f"./{CONFIG['progress_file']}"
            with open(local_path, 'w') as f:
                json.dump(clean_data, f, indent=2)
            
            # Upload to repository
            upload_file(
                path_or_fileobj=local_path,
                path_in_repo=CONFIG['progress_file'],
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Progress update: {self.progress_data['total_processed']} processed"
            )
            print(f"💾 Progress saved to repository")
        except Exception as e:
            print(f"❌ Failed to save progress: {e}")
    
    def add_processed(self, index, variants_count=0, quality_stats=None):
        """Mark index as processed"""
        if index not in self.progress_data['processed_indices']:
            self.progress_data['processed_indices'].append(index)
            self.progress_data['total_processed'] += 1
            self.progress_data['total_variants_created'] += variants_count
        
        if quality_stats:
            self.update_statistics(quality_stats)
    
    def add_error(self, error_data):
        """Add error to progress tracking"""
        error_entry = {
            'timestamp': datetime.now().isoformat(),
            **error_data
        }
        self.progress_data['errors'].append(error_entry)
        
        # Keep only last 100 errors to prevent file bloat
        if len(self.progress_data['errors']) > 100:
            self.progress_data['errors'] = self.progress_data['errors'][-100:]
    
    def update_statistics(self, stats):
        """Update running statistics"""
        current_stats = self.progress_data['statistics']
        
        # Update quality counts
        for quality, count in stats.get('quality_counts', {}).items():
            current_stats['quality_counts'][quality] += count
        
        # Update averages (simple running average)
        total_variants = self.progress_data['total_variants_created']
        if total_variants > 0:
            current_stats['avg_score'] = (
                (current_stats['avg_score'] * (total_variants - stats.get('variants_count', 0)) + 
                 stats.get('avg_score', 0) * stats.get('variants_count', 0)) / total_variants
            )
            current_stats['avg_word_count'] = (
                (current_stats['avg_word_count'] * (total_variants - stats.get('variants_count', 0)) + 
                 stats.get('avg_word_count', 0) * stats.get('variants_count', 0)) / total_variants
            )
    
    def is_processed(self, index):
        """Check if index is already processed"""
        return index in self.progress_data['processed_indices']
    
    def get_next_chunk_start(self, total_rows):
        """Get the next completely unprocessed chunk start"""
        self.progress_data['total_chunks'] = (total_rows + CONFIG['chunk_size'] - 1) // CONFIG['chunk_size']
        
        # Find the first chunk where ALL items are unprocessed
        for chunk_start in range(0, total_rows, CONFIG['chunk_size']):
            chunk_end = min(chunk_start + CONFIG['chunk_size'], total_rows)
            chunk_indices = list(range(chunk_start, chunk_end))
            
            # Check if ALL indices in this chunk are unprocessed
            all_unprocessed = all(not self.is_processed(idx) for idx in chunk_indices)
            if all_unprocessed:
                return chunk_start
        
        # If no completely unprocessed chunk, find the first partially processed chunk
        for chunk_start in range(0, total_rows, CONFIG['chunk_size']):
            chunk_end = min(chunk_start + CONFIG['chunk_size'], total_rows)
            chunk_indices = list(range(chunk_start, chunk_end))
            
            # Check if any index in this chunk is unprocessed
            any_unprocessed = any(not self.is_processed(idx) for idx in chunk_indices)
            if any_unprocessed:
                return chunk_start
        
        return None  # All processed
    
    def show_progress(self, total_rows):
        """Display current progress"""
        processed = len(self.progress_data['processed_indices'])
        remaining = total_rows - processed
        progress_pct = (processed / total_rows) * 100 if total_rows > 0 else 0
        
        print(f"Progress: {processed:,}/{total_rows:,} ({progress_pct:.1f}%)")
        print(f"Remaining: {remaining:,}")
        print(f"Total variants created: {self.progress_data['total_variants_created']:,}")
        print(f"API requests made: {self.progress_data['request_count']:,}")
        
        # Find next unprocessed chunk
        next_chunk_start = self.get_next_chunk_start(total_rows)
        if next_chunk_start is not None:
            next_chunk_num = next_chunk_start // CONFIG['chunk_size'] + 1
            total_chunks = (total_rows + CONFIG['chunk_size'] - 1) // CONFIG['chunk_size']
            print(f"Next chunk to process: {next_chunk_num}/{total_chunks} (starting at index {next_chunk_start})")
        else:
            print("All chunks processed!")
        
        return next_chunk_start

In [ ]:
# Cell 6: Dataset Loading
dataset = load_dataset(CONFIG['original_dataset'])
if isinstance(dataset, dict):
    ds = dataset['train'] if 'train' in dataset else dataset[list(dataset.keys())[0]]
else:
    ds = dataset

print(f"Dataset: {len(ds)} rows")
print(f"Columns: {list(ds[0].keys())}")

# Initialize progress manager
progress_manager = ProgressManager()

In [ ]:
# Cell 7: Clean Function (unchanged)
def clean(example):
    text = example['Answer']

    # Remove markdown links: [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Replace [[5]](#_ftn5 "_ftnref5") with [5]
    text = re.sub(r'\[\[(\d+)\]\]\(#_ftn\d+.*?\)', r'[\1]', text)

    # Remove footnote anchors like [1](#ref)
    text = re.sub(r'\[\d+\]\(#.*?\)', '', text)

    # Remove heading + date + image pattern block
    text = re.sub(
        r'####.*?\n\s*\d{1,2}\s+\w+,\s+\d{4}\s*\n\s*!\[.*?\]\(.*?\)\s*',
        '',
        text,
        flags=re.DOTALL
    )

    # Remove block between "## ULASAN LENGKAP" and "### KLINIK TERKAIT"
    text = re.sub(
        r'## ULASAN LENGKAP.*?### KLINIK TERKAIT',
        '',
        text,
        flags = re.DOTALL
    )

    # Remove lines like: Baca juga: **Some Title**
    text = re.sub(r'Baca juga:\s+\*\*.*?\*\*', '', text)

    # Remove base64-encoded inline images
    text = re.sub(r'!\[\]\(data:image\/[a-zA-Z]+;base64,[^\)]+\)', '', text)

    # Promotional/legal education phrases
    text = re.sub(
        r'Belajar Hukum Secara Online dari Pengajar Berkompeten Dengan Biaya TerjangkauMulai DariRp\.?\s?149\.000Lihat Semua Kelas',
        '',
        text
    )
    text = re.sub(
        r'Penjelasan lebih lanjut dapat Anda baca ulasan di bawah ini.',
        '',
        text
    )

    # Hukumonline Pro promotion
    text = re.sub(
        r'Perkaya riset hukum Anda dengan analisis hukum terbaru.*?di sini\*\*\.',
        '',
        text,
        flags = re.IGNORECASE
    )

    # Closing statement
    text = re.sub(
        r'Demikian jawaban dari kami, semoga bermanfaat\.',
        '',
        text,
        flags = re.IGNORECASE
    )

    # Remove disclaimer/legal notes and editorial notes (partial fuzzy match using key phrases)
    disclaimer_patterns = [
        r'Artikel di bawah ini adalah pemutakhiran.*?dipublikasikan.*?\.',
        r'Seluruh informasi hukum.*?untuk tujuan pendidikan.*?\.',
        r'konsultasikan langsung dengan \*\*Konsultan Mitra Justika\*\*',
        r'kami memiliki keterbatasan informasi.*?Hal tersebut tentu berdampak.*?jawaban.*?'
    ]
    for pattern in disclaimer_patterns:
        text = re.sub(pattern, '', text, flags = re.IGNORECASE | re.DOTALL)

    # Normalize spacing (but keep \n)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' *\n *', '\n', text)
    text = re.sub(r'\n{2,}', '\n', text)

    return {'Answer': text}

cleaned_ds = ds.map(clean, desc="Cleaning text")
print(f"Dataset cleaned: {len(cleaned_ds)} rows")

In [ ]:
# Cell 8: Legal Knowledge Base (unchanged)
class LegalKB:
    def __init__(self):
        self.legal_terms = {
            'hukum_pidana': ['pidana', 'kriminal', 'kejahatan', 'pelanggaran', 'sanksi'],
            'hukum_perdata': ['perdata', 'kontrak', 'perjanjian', 'ganti rugi'],
            'hukum_tata_negara': ['konstitusi', 'UUD', 'pemerintahan', 'negara'],
            'lembaga': ['mahkamah', 'pengadilan', 'kejaksaan', 'kepolisian']
        }
        
        self.citation_patterns = [
            r'Pasal\s+\d+', r'UU\s+(No\.|Nomor)\s*\d+', r'PP\s+(No\.|Nomor)\s*\d+',
            r'KUHP\s+Pasal\s+\d+', r'Mahkamah\s+Agung'
        ]
    
    def extract_citations(self, text):
        citations = []
        for pattern in self.citation_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            citations.extend(matches)
        return list(set(citations))
    
    def count_legal_terms(self, text):
        text_lower = text.lower()
        term_counts = {}
        for category, terms in self.legal_terms.items():
            count = sum(1 for term in terms if term in text_lower)
            term_counts[category] = count
        return term_counts

legal_kb = LegalKB()

In [ ]:
# Cell 9: Scorer (unchanged)
class Scorer:
    def __init__(self):
        self.legal_kb = LegalKB()
        self.tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
        self.cache = {}
        self.error_log = []
    
    def semantic_similarity(self, text1, text2):
        try:
            texts = [text1, text2]
            tfidf_matrix = self.tfidf.fit_transform(texts)
            similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
            return float(similarity)
        except:
            return 0.0
    
    def citation_preservation(self, orig, gen):
        try:
            orig_cites = set(self.legal_kb.extract_citations(orig))
            gen_cites = set(self.legal_kb.extract_citations(gen))
            
            if not orig_cites:
                return 1.0
            
            preserved = orig_cites.intersection(gen_cites)
            return len(preserved) / len(orig_cites)
        except:
            return 0.0
    
    def legal_terminology(self, orig, gen):
        try:
            orig_terms = self.legal_kb.count_legal_terms(orig)
            gen_terms = self.legal_kb.count_legal_terms(gen)
            
            total_orig = sum(orig_terms.values())
            if total_orig == 0:
                return 1.0
            
            preserved_score = 0.0
            for category, orig_count in orig_terms.items():
                if orig_count > 0:
                    gen_count = gen_terms.get(category, 0)
                    ratio = min(1.0, gen_count / orig_count)
                    preserved_score += ratio * (orig_count / total_orig)
            
            return preserved_score
        except:
            return 0.0
    
    def calculate_scores(self, orig_data, gen_data):
        try:
            orig_q = orig_data.get(CONFIG['question_column'], '')
            orig_a = orig_data.get(CONFIG['answer_column'], '')
            gen_q = gen_data.get('question', '')
            gen_a = gen_data.get('answer', '')
            
            cache_key = hashlib.md5(f"{orig_q[:100]}{gen_q[:100]}".encode()).hexdigest()
            if cache_key in self.cache:
                return self.cache[cache_key]
            
            scores = {
                'semantic_similarity_q': self.semantic_similarity(orig_q, gen_q),
                'semantic_similarity_a': self.semantic_similarity(orig_a, gen_a),
                'citation_preservation': self.citation_preservation(f"{orig_q} {orig_a}", f"{gen_q} {gen_a}"),
                'legal_terminology': self.legal_terminology(f"{orig_q} {orig_a}", f"{gen_q} {gen_a}"),
            }
            
            weights = {
                'semantic_similarity_q': 0.25,
                'semantic_similarity_a': 0.25,
                'citation_preservation': 0.25,
                'legal_terminology': 0.25,
            }
            
            overall = sum(scores[k] * weights[k] for k in weights)
            scores['overall_score'] = overall
            
            action = 'keep' if overall >= CONFIG['min_overall_score'] else 'revise' if overall >= 0.4 else 'reject'
            scores['recommendation'] = action
            
            self.cache[cache_key] = scores
            return scores
            
        except Exception as e:
            return {
                'semantic_similarity_q': 0.0,
                'semantic_similarity_a': 0.0,
                'citation_preservation': 0.0,
                'legal_terminology': 0.0,
                'overall_score': 0.0,
                'recommendation': 'error'
            }

scorer = Scorer()

In [ ]:
# Cell 10: Modified Synthesizer
class Synthesizer:
    def __init__(self, progress_manager):
        self.model = genai.GenerativeModel(
            CONFIG['model_name'],
            generation_config=genai.types.GenerationConfig(
                temperature=CONFIG['temperature'],
                max_output_tokens=CONFIG['output_token_limit'](),
                top_p=0.8,
                top_k=40
            )
        )
        self.scorer = Scorer()
        self.progress_manager = progress_manager
        self.request_count = progress_manager.progress_data['request_count']
    
    def log_error(self, error_type, message, row_index=None, example_data=None, api_response=None):
        error_entry = {
            'error_type': error_type,
            'message': message,
            'row_index': row_index,
            'request_count': self.request_count,
            'example_preview': {
                'question': str(example_data.get(CONFIG['question_column'], ''))[:100] if example_data else None,
                'answer': str(example_data.get(CONFIG['answer_column'], ''))[:100] if example_data else None,
            } if example_data else None,
            'api_response_preview': str(api_response)[:200] if api_response else None
        }
        self.progress_manager.add_error(error_entry)
        print(f"    Logged Error: {error_type} - {message}")
    
    def create_prompt(self, question, answer, context=None):
        q_words = len(question.split())
        a_words = len(answer.split())
        
        context_part = f"\nKonteks: {context[:800]}" if context else ""
        
        return f"""Anda adalah ahli hukum Indonesia senior. Buat {CONFIG['num_variants']} variasi berkualitas tinggi dari Q&A hukum berikut:

MATERI ASLI:{context_part}
Q: {question}
A: {answer}

PERSYARATAN:
1. Setiap jawaban minimal {max(a_words, 50)} kata
2. Pertahankan SEMUA kutipan hukum (Pasal, UU, PP, dll.)
3. Pertahankan terminologi hukum Indonesia
4. Gunakan bahasa formal dan analisis mendalam
5. Setiap variasi berbeda pendekatan tapi substansi sama

VARIASI:
- Akademik: analisis teoretis mendalam
- Praktis: aplikasi dalam praktik hukum
- Komparatif: hubungan dengan peraturan lain

FORMAT JSON:
[""" + ",\n".join([f"""  {{"question": "variasi {i+1}", "answer": "jawaban lengkap {i+1}"}}""" for i in range(CONFIG['num_variants'])]) + f"""
]

Berikan HANYA JSON array yang valid."""
    
    def categorize_quality(self, scores):
        overall = scores['overall_score']
        if overall >= 0.85:
            return 'premium'
        elif overall >= 0.75:
            return 'high'
        elif overall >= 0.65:
            return 'medium'
        elif overall >= 0.5:
            return 'low'
        else:
            return 'poor'
    
    def assess_content_metrics(self, original_answer, generated_answer):
        orig_words = len(original_answer.split())
        gen_words = len(generated_answer.split())
        
        metrics = {
            'length_ratio': gen_words / orig_words if orig_words > 0 else 0,
            'word_count_original': orig_words,
            'word_count_generated': gen_words,
            'char_count_generated': len(generated_answer),
            'is_substantial': gen_words >= 50,
            'is_comprehensive': gen_words >= orig_words * 0.7,
            'has_legal_terms': any(term in generated_answer.lower() for term in 
                                 ['pasal', 'undang-undang', 'peraturan', 'hukum', 'berdasarkan']),
        }
        
        words = generated_answer.lower().split()
        unique_words = set(words)
        metrics['vocabulary_richness'] = len(unique_words) / len(words) if len(words) > 0 else 0
        metrics['is_repetitive'] = metrics['vocabulary_richness'] < 0.6
        
        return metrics
    
    def synthesize(self, example, row_index=None):
        if self.progress_manager.is_processed(row_index):
            return None
        
        question = str(example.get(CONFIG['question_column'], ''))
        answer = str(example.get(CONFIG['answer_column'], ''))
        context = str(example.get(CONFIG['context_column'], '')) if CONFIG['context_column'] else None
        
        if len(question) < 10 or len(answer) < 20:
            self.log_error('content_short', f"Q:{len(question)} chars, A:{len(answer)} chars", row_index, example)
        
        total_chars = len(question) + len(answer) + (len(context) if context else 0)
        estimated_tokens = total_chars // 4
        
        if estimated_tokens > CONFIG['input_token_limit'] // 2:
            question = question[:800]
            answer = answer[:1500]
            context = context[:400] if context else None
            print(f"    Content truncated for processing")
        
        prompt = self.create_prompt(question, answer, context)
        
        try:
            time.sleep(4)
            
            response = self.model.generate_content(prompt)
            self.request_count += 1
            self.progress_manager.progress_data['request_count'] = self.request_count
            
            if not response.candidates:
                self.log_error('api_no_candidates', 'No candidates returned', row_index, example)
                return None
            
            candidate = response.candidates[0]
            
            if candidate.finish_reason == 2:
                self.log_error('api_safety_filter', 'Content filtered by safety', row_index, example)
                return None
            elif candidate.finish_reason == 3:
                self.log_error('api_recitation', 'Content flagged as recitation', row_index, example)
                return None
            elif candidate.finish_reason == 4:
                self.log_error('api_other_error', 'Other API issue', row_index, example)
                return None
            
            if not candidate.content or not candidate.content.parts:
                self.log_error('api_no_content', 'No content in response', row_index, example)
                return None
            
            response_text = candidate.content.parts[0].text
            
            if not response_text or not response_text.strip():
                self.log_error('api_empty_response', 'Empty response text', row_index, example)
                return None
            
            content = response_text.strip()
            json_content = None
            
            if "```json" in content:
                try:
                    json_content = content.split("```json")[1].split("```")[0].strip()
                except:
                    pass
            
            if not json_content and "```" in content:
                try:
                    json_content = content.split("```")[1].strip()
                except:
                    pass
            
            if not json_content:
                json_match = re.search(r'\[.*?\]', content, re.DOTALL)
                if json_match:
                    json_content = json_match.group(0)
                else:
                    self.log_error('json_extraction_failed', 'No JSON found', row_index, example, content[:300])
                    return None
            
            json_content = json_content.replace('\n', ' ')
            json_content = re.sub(r'\s+', ' ', json_content)
            json_content = re.sub(r',\s*}', '}', json_content)
            json_content = re.sub(r',\s*]', ']', json_content)
            
            try:
                variants = json.loads(json_content)
            except json.JSONDecodeError as e:
                self.log_error('json_parse_error', f"JSONDecodeError: {str(e)}", row_index, example, json_content[:200])
                return None
            
            if not isinstance(variants, list):
                self.log_error('json_not_list', f"Response is {type(variants)}", row_index, example)
                return None
            
            results = []
            quality_counts = {'premium': 0, 'high': 0, 'medium': 0, 'low': 0, 'poor': 0}
            
            for i, variant in enumerate(variants):
                if not isinstance(variant, dict) or 'question' not in variant or 'answer' not in variant:
                    self.log_error('variant_invalid', f"Variant {i} invalid", row_index, example)
                    continue
                
                clean_q = str(variant['question']).strip()
                clean_a = str(variant['answer']).strip()
                
                quality_flag = 'too_short' if len(clean_q) < 5 or len(clean_a) < 10 else 'acceptable'
                
                clean_variant = {'question': clean_q, 'answer': clean_a}
                
                try:
                    scores = self.scorer.calculate_scores(example, clean_variant)
                except Exception as e:
                    self.log_error('scoring_error', f"Scoring failed for variant {i}: {str(e)}", row_index, example)
                    scores = {
                        'overall_score': 0.0,
                        'semantic_similarity_q': 0.0,
                        'semantic_similarity_a': 0.0,
                        'citation_preservation': 0.0,
                        'legal_terminology': 0.0,
                        'recommendation': 'error'
                    }
                
                content_metrics = self.assess_content_metrics(answer, clean_a)
                quality_category = self.categorize_quality(scores)
                quality_counts[quality_category] += 1
                
                result = {
                    'original_question': question,
                    'original_answer': answer,
                    'generated_question': clean_variant['question'],
                    'generated_answer': clean_variant['answer'],
                    'overall_score': scores['overall_score'],
                    'semantic_similarity_q': scores['semantic_similarity_q'],
                    'semantic_similarity_a': scores['semantic_similarity_a'],
                    'citation_preservation': scores['citation_preservation'],
                    'legal_terminology': scores['legal_terminology'],
                    'recommendation': scores['recommendation'],
                    'quality_category': quality_category,
                    'quality_flag': quality_flag,
                    'original_word_count': content_metrics['word_count_original'],
                    'generated_word_count': content_metrics['word_count_generated'],
                    'generated_char_count': content_metrics['char_count_generated'],
                    'length_ratio': content_metrics['length_ratio'],
                    'vocabulary_richness': content_metrics['vocabulary_richness'],
                    'is_substantial': content_metrics['is_substantial'],
                    'is_comprehensive': content_metrics['is_comprehensive'],
                    'has_legal_terms': content_metrics['has_legal_terms'],
                    'is_repetitive': content_metrics['is_repetitive'],
                    'variant_number': i + 1,
                    'row_index': row_index,
                    'timestamp': datetime.now().isoformat()
                }
                
                for col in CONFIG['preserve_columns']:
                    if col in example:
                        result[f'original_{col}'] = example[col]
                
                results.append(result)
            
            if results:
                # Mark as processed and update statistics
                total_variants = len(results)
                avg_score = sum(r['overall_score'] for r in results) / total_variants
                avg_word_count = sum(r['generated_word_count'] for r in results) / total_variants
                
                quality_stats = {
                    'quality_counts': quality_counts,
                    'avg_score': avg_score,
                    'avg_word_count': avg_word_count,
                    'variants_count': total_variants
                }
                
                self.progress_manager.add_processed(row_index, total_variants, quality_stats)
                
                print(f"    Success: {total_variants} variants stored")
                print(f"    Quality: {quality_counts}")
                print(f"    Avg score: {avg_score:.3f}")
                return results
            else:
                self.log_error('no_valid_variants', 'No valid variants produced', row_index, example)
                return None
            
        except Exception as e:
            self.log_error('unexpected_error', f"Unexpected error: {str(e)}", row_index, example)
            return None

synthesizer = Synthesizer(progress_manager)

In [ ]:
# Cell 11: Modified Progress Check Function
def show_progress():
    next_chunk = progress_manager.show_progress(len(cleaned_ds))
    return next_chunk

In [ ]:
# Cell 12: Modified Run Single Chunk (MAIN PROCESSING CELL)
def process_chunk(chunk_start, chunk_size=None):
    if chunk_size is None:
        chunk_size = CONFIG['chunk_size']
    
    chunk_end = min(chunk_start + chunk_size, len(cleaned_ds))
    total_chunks = (len(cleaned_ds) + CONFIG['chunk_size'] - 1) // CONFIG['chunk_size']
    current_chunk = chunk_start // CONFIG['chunk_size'] + 1
    
    # Update progress manager current chunk
    progress_manager.progress_data['current_chunk'] = current_chunk
    progress_manager.progress_data['total_chunks'] = total_chunks
    
    print(f"Processing chunk {current_chunk}/{total_chunks} (indices {chunk_start}-{chunk_end-1})")
    
    all_results = []
    processed_count = 0
    
    for i in range(chunk_start, chunk_end):
        idx = i
        
        # Skip if already processed
        if progress_manager.is_processed(idx):
            print(f"  {i-chunk_start+1}/{chunk_end-chunk_start}: Index {idx} - Already processed, skipping")
            continue
            
        print(f"  {i-chunk_start+1}/{chunk_end-chunk_start}: Index {idx}")
        
        example = cleaned_ds[idx]
        
        if isinstance(example, dict):
            example_dict = example
        elif hasattr(example, 'keys'):
            example_dict = {key: example[key] for key in example.keys()}
        else:
            try:
                example_dict = dict(example)
            except:
                continue
        
        example_dict['processing_row_index'] = idx
        
        results = synthesizer.synthesize(example_dict, row_index=idx)
        
        if results:
            for result in results:
                result['original_index'] = idx
                result['chunk_number'] = current_chunk
            all_results.extend(results)
            processed_count += 1
        
        # Save progress after each item
        try:
            progress_manager.save_progress()
        except:
            pass
    
    # Upload chunk as raw parquet file with your exact naming
    if all_results:
        try:
            df = pd.DataFrame(all_results)
            
            # Filter to essential columns
            essential_columns = [
                'original_question', 'original_answer',
                'generated_question', 'generated_answer',
                'overall_score', 'semantic_similarity_q', 'semantic_similarity_a',
                'citation_preservation', 'legal_terminology', 'recommendation',
                'variant_number', 'row_index', 'timestamp',
                'original_index', 'chunk_number'
            ]
            
            existing_columns = [col for col in essential_columns if col in df.columns]
            upload_df = df[existing_columns].copy()
            
            # Save as parquet and upload with your exact naming
            filename = f"train-{current_chunk:05d}-of-10000.parquet"
            temp_filepath = f"/tmp/{filename}"
            upload_df.to_parquet(temp_filepath, index=False)
            
            # Upload raw parquet file
            upload_file(
                path_or_fileobj=temp_filepath,
                path_in_repo=f"./data/{filename}",
                repo_id=CONFIG['output_repository'],
                repo_type="dataset",
                commit_message=f"Chunk {current_chunk}/10000: {len(all_results)} variants"
            )
            
            print(f"Uploaded {filename} to repository ({len(all_results)} entries)")
            
            # Clean up temp file
            try:
                os.remove(temp_filepath)
            except:
                pass
            
            avg_score = df['overall_score'].mean()
            high_quality = (df['overall_score'] >= CONFIG['min_overall_score']).sum()
            print(f"Avg score: {avg_score:.3f}, High quality: {high_quality}/{len(df)}")
            
        except Exception as e:
            print(f"  ⚠️ Upload failed for chunk {current_chunk}: {e}")
    
    print(f"  Summary: {processed_count} processed")
    
    # Final progress save
    try:
        progress_manager.save_progress()
    except:
        pass
    
    return all_results

In [ ]:
# Cell 13: Auto-continue next chunk
def continue_processing():
    """Continue processing from where we left off"""
    next_chunk_start = show_progress()
    
    if next_chunk_start is not None:
        total_chunks = (len(cleaned_ds) + CONFIG['chunk_size'] - 1) // CONFIG['chunk_size']
        current_chunk = next_chunk_start // CONFIG['chunk_size'] + 1
        
        print(f"\nAuto-continuing from chunk {current_chunk}/{total_chunks}")
        results = process_chunk(next_chunk_start, CONFIG['chunk_size'])
        
        # After processing, check if there are more chunks
        next_next_chunk = progress_manager.get_next_chunk_start(len(cleaned_ds))
        if next_next_chunk is not None:
            print(f"Next unprocessed chunk: {next_next_chunk // CONFIG['chunk_size'] + 1}")
        else:
            print("All chunks completed!")
        
        return None  # Don't return results to avoid printing
    else:
        print("All chunks have been processed!")
        return None

# Auto-continue processing
print("Checking progress and continuing...")
continue_processing()

In [ ]:
# Cell 14: Check Processing Status
def check_completion_status():
    """Check processing completion status"""
    try:
        # Load dataset to check splits
        dataset_info = load_dataset(CONFIG['output_repository'])
        splits = list(dataset_info.keys())
        
        total_chunks = (len(cleaned_ds) + CONFIG['chunk_size'] - 1) // CONFIG['chunk_size']
        chunk_splits = [s for s in splits if s.startswith('train') and 'of' in s]
        
        print(f"📊 PROCESSING STATUS")
        print(f"=" * 40)
        print(f"Expected chunks: {total_chunks}")
        print(f"Completed chunks: {len(chunk_splits)}")
        print(f"Progress: {len(chunk_splits)}/{total_chunks} ({len(chunk_splits)/total_chunks*100:.1f}%)")
        
        if chunk_splits:
            print(f"Latest splits: {sorted(chunk_splits)[-3:]}")
        
        if len(chunk_splits) >= total_chunks:
            print("🎉 All chunks completed!")
            print(f"✅ Dataset available at: {CONFIG['output_repository']}")
            print("📋 Each chunk is a separate split for easy access")
        else:
            missing = total_chunks - len(chunk_splits)
            print(f"⚠️ {missing} chunks remaining")
        
        return len(chunk_splits) >= total_chunks
        
    except Exception as e:
        print(f"❌ Status check failed: {e}")
        return False

# Check current status
#check_completion_status()

In [ ]:
# Cell 15: Utility Functions
def analyze_quality():
    """Analyze quality distribution of current results"""
    local_files = [f for f in os.listdir(CONFIG['data_dir']) if f.endswith('.parquet')]
    
    if not local_files:
        print("No local data files found")
        return
    
    all_data = []
    for file in local_files:
        filepath = os.path.join(CONFIG['data_dir'], file)
        df = pd.read_parquet(filepath)
        all_data.append(df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    
    print("QUALITY ANALYSIS")
    print("=" * 30)
    print(f"Total variants: {len(combined_df):,}")
    
    # Quality distribution
    quality_dist = combined_df['quality_category'].value_counts()
    print(f"\nQuality Distribution:")
    for quality, count in quality_dist.items():
        pct = (count / len(combined_df)) * 100
        print(f"  {quality}: {count:,} ({pct:.1f}%)")
    
    # Score statistics
    print(f"\nScore Statistics:")
    print(f"  Average overall score: {combined_df['overall_score'].mean():.3f}")
    print(f"  Median overall score: {combined_df['overall_score'].median():.3f}")
    print(f"  High quality (≥{CONFIG['min_overall_score']}): {(combined_df['overall_score'] >= CONFIG['min_overall_score']).sum():,}")
    
    # Content statistics
    print(f"\nContent Statistics:")
    print(f"  Avg generated words: {combined_df['generated_word_count'].mean():.1f}")
    print(f"  Avg length ratio: {combined_df['length_ratio'].mean():.2f}")
    print(f"  Substantial content: {combined_df['is_substantial'].sum():,}")
    print(f"  Has legal terms: {combined_df['has_legal_terms'].sum():,}")
    
    return combined_df

def update_variant_count(new_count):
    """Update the number of variants to generate"""
    CONFIG['num_variants'] = new_count
    CONFIG['output_token_limit'] = lambda: CONFIG['num_variants'] * 4000
    
    # Reinitialize synthesizer with new settings
    global synthesizer
    synthesizer = Synthesizer(progress_manager)
    
    print(f"✅ Updated to generate {new_count} variants per original")
    print(f"✅ Output token limit updated to {CONFIG['output_token_limit']()} tokens")

def reset_progress():
    """Reset all progress (use with caution!)"""
    confirm = input("⚠️  This will reset ALL progress. Type 'CONFIRM' to proceed: ")
    if confirm == 'CONFIRM':
        progress_manager.progress_data = {
            'processed_indices': [],
            'current_chunk': 0,
            'total_chunks': 0,
            'total_processed': 0,
            'total_variants_created': 0,
            'request_count': 0,
            'start_time': datetime.now().isoformat(),
            'last_update': None,
            'errors': [],
            'statistics': {
                'quality_counts': {'premium': 0, 'high': 0, 'medium': 0, 'low': 0, 'poor': 0},
                'avg_score': 0.0,
                'avg_word_count': 0.0
            }
        }
        progress_manager.save_progress()
        print("✅ Progress reset complete")
    else:
        print("❌ Reset cancelled")

def view_recent_errors(limit=10):
    """View recent errors from progress tracking"""
    errors = progress_manager.progress_data.get('errors', [])
    if not errors:
        print("No errors found in progress tracking")
        return
    
    recent_errors = errors[-limit:]
    print(f"📋 Last {len(recent_errors)} errors:")
    print("=" * 50)
    
    for i, error in enumerate(recent_errors, 1):
        print(f"{i}. [{error.get('timestamp', 'Unknown')}]")
        print(f"   Type: {error.get('error_type', 'Unknown')}")
        print(f"   Message: {error.get('message', 'No message')}")
        print(f"   Row: {error.get('row_index', 'Unknown')}")
        print("-" * 30)

print("✅ Complete Indonesian Legal Q&A Synthesis Pipeline Ready!")
print("🔄 Cloud-to-Cloud Pipeline with Progress Tracking")
print("\nKey Functions:")
print("- show_progress() - Check processing status")
print("- run_next_chunk() - Process next unprocessed chunk")
print("- analyze_quality() - Analyze quality distribution") 
print("- update_variant_count(n) - Change number of variants")
print("- combine_and_upload() - Final upload to repository")
print("- view_recent_errors(n) - View recent errors")
print("- reset_progress() - Reset all progress (caution!)")
print(f"\nCurrent settings: {CONFIG['num_variants']} variants, {CONFIG['output_token_limit']()} max tokens")
print(f"Output naming: train-XXXXX-of-XXXXX.parquet")

# Auto-show progress on startup
show_progress()

In [ ]:
# Cell 16: Repository Cleanup
def clean_repository_splits():
    """Clean up conflicting splits in the repository"""
    try:
        from huggingface_hub import HfApi, delete_file
        
        api = HfApi()
        
        # List all files in the repository
        try:
            repo_files = api.list_repo_files(
                repo_id=CONFIG['output_repository'],
                repo_type="dataset"
            )
            
            # Delete problematic split files
            splits_to_clean = ['full', 'high_quality', 'medium_plus']
            
            for split in splits_to_clean:
                files_to_delete = [f for f in repo_files if f.startswith(f"{split}/") or f.startswith(f"data/{split}-")]
                
                for file_path in files_to_delete:
                    try:
                        delete_file(
                            path_in_repo=file_path,
                            repo_id=CONFIG['output_repository'],
                            repo_type="dataset",
                            commit_message=f"Clean up conflicting split: {split}"
                        )
                        print(f"🗑️ Deleted: {file_path}")
                    except Exception as e:
                        print(f"⚠️ Could not delete {file_path}: {e}")
            
            print("✅ Repository splits cleaned")
            
        except Exception as e:
            print(f"⚠️ Could not list repository files: {e}")
            
    except Exception as e:
        print(f"❌ Failed to clean repository: {e}")
        print("💡 Manual fix: Go to your Hugging Face repository and delete the conflicting splits")

# Run the cleanup
print("🧹 Cleaning repository splits...")
clean_repository_splits()